**CI twin of `ch12-gradient-boosting.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
from lib.data import load_csv
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

homes = load_csv("california-housing-sample")
feats = ["MedInc", "HouseAge", "AveRooms", "AveBedrms",
         "Population", "AveOccup", "Latitude", "Longitude"]
X, y = homes[feats], homes["MedHouseVal"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)

lr = 0.5                                   # how much of each fix we accept
pred_tr = np.full(len(ytr), ytr.mean())    # round 0: predict the mean
pred_te = np.full(len(yte), ytr.mean())
print(f"round 0 (the mean): train MAE {mean_absolute_error(ytr, pred_tr):.3f}"
      f"   test MAE {mean_absolute_error(yte, pred_te):.3f}")

for rnd in range(1, 6):
    residuals = ytr - pred_tr              # what the team still gets wrong
    fixer = DecisionTreeRegressor(max_depth=2, random_state=0)
    fixer.fit(Xtr, residuals)              # the new tree studies ONLY that
    pred_tr = pred_tr + lr * fixer.predict(Xtr)
    pred_te = pred_te + lr * fixer.predict(Xte)
    print(f"round {rnd}:            train MAE "
          f"{mean_absolute_error(ytr, pred_tr):.3f}   test MAE "
          f"{mean_absolute_error(yte, pred_te):.3f}")

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import HistGradientBoostingRegressor

scores = -cross_val_score(HistGradientBoostingRegressor(random_state=0),
                          X, y, cv=5, scoring="neg_mean_absolute_error")
print(f"boosted CV MAE: {scores.mean():.3f} ± {scores.std():.3f}")
print("forest (Ch11):  0.532 ± 0.050")
print("linear (Ch4):   0.558 ± 0.031")

In [ ]:
print("max_iter   train MAE   test MAE")
for iters in (10, 50, 100, 300, 1000):
    m = HistGradientBoostingRegressor(max_iter=iters,
                                      random_state=0).fit(Xtr, ytr)
    print(f"  {iters:4}      {mean_absolute_error(ytr, m.predict(Xtr)):.3f}"
          f"      {mean_absolute_error(yte, m.predict(Xte)):.3f}")

In [ ]:
import xgboost as xgb

print(f"xgboost {xgb.__version__}, in the browser\n")

xgb_scores = -cross_val_score(
    xgb.XGBRegressor(random_state=0, verbosity=0),
    X, y, cv=5, scoring="neg_mean_absolute_error")
print(f"XGBoost CV MAE (default dials): {xgb_scores.mean():.3f} "
      f"± {xgb_scores.std():.3f}")

In [ ]:
from sklearn.metrics import accuracy_score

pg = load_csv("penguins").dropna(subset=["bill_length_mm",
                                         "flipper_length_mm"])
yp = pg["species"].map({"Adelie": 0, "Chinstrap": 1, "Gentoo": 2})
pf = ["flipper_length_mm", "bill_length_mm"]
ptr, pte, ltr, lte = train_test_split(pg[pf], yp, test_size=0.25,
                                      random_state=42, stratify=yp)

clf = xgb.XGBClassifier(random_state=0, verbosity=0).fit(ptr, ltr)
print(f"held-out accuracy: {accuracy_score(lte, clf.predict(pte)):.3f}")

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

xs = [[1.0], [2.0], [3.0], [4.0]]
ys = np.array([1.0, 2.0, 2.0, 3.0])

base = np.full(4, ys.mean())
residuals = ys - base
fixer = DecisionTreeRegressor(max_depth=1, random_state=0).fit(xs, residuals)
boosted = base + 0.5 * fixer.predict(xs)

run_tests([
    ("first house corrected downward", round(float(boosted[0]), 3), 1.5),
    ("round-1 MAE", round(float(mean_absolute_error(ys, boosted)), 4), 0.4167),
])

In [ ]:
def residuals(y_true, y_pred):
    return [t - p for t, p in zip(y_true, y_pred)]

def boost_predict(tree_preds, lr):
    return [sum(lr * p for p in column) for column in zip(*tree_preds)]

run_tests([
    ("what's still wrong", residuals([3.0, 1.0], [2.5, 1.5]), [0.5, -0.5]),
    ("perfect team, no residuals", residuals([2.0, 2.0], [2.0, 2.0]),
     [0.0, 0.0]),
    ("three fixes, summed and scaled",
     boost_predict([[1.0, 2.0], [0.5, -0.5], [0.2, 0.2]], 0.1),
     [0.17, 0.17]),
    ("lr of 1 accepts fixes whole",
     boost_predict([[1.0], [2.0]], 1.0), [3.0]),
], tol=1e-9)